# Supplementary Demo: Topic Modeling with LDA

**CPE 393 — Text Analytics | Guided demonstration (not an assignment)**

This notebook builds on the original State of the Union speeches example (1970–2012). You can run each cell in order, read the explanations, and experiment with the parameters at your own pace. **Nothing needs to be submitted.**

**By the end, you should understand:**
- How a collection of documents is converted into the bag-of-words (BoW) representation used by LDA.
- Why LDA produces *topics as distributions over words* and *documents as mixtures of topics*.
- How the number of topics and preprocessing choices affect interpretation.
- How to interpret the output and recognize limitations rather than assume every topic is meaningful.

**Suggested approach:** Read the short *Think about it* prompts before running a cell, then compare your expectation with the output. These are discussion/reflection prompts, **not homework**.

## 1. Load the dataset

The dataset contains U.S. presidential **State of the Union** addresses from 1970–2012. Each speech is treated as one document. We will use the same CSV source as the original demo.

**Environment:** The notebook works in Google Colab or a local Jupyter environment with internet access for the initial download. Google Drive mounting is **not required**.

In [1]:
# Install dependencies if needed (uncomment and run in a new environment).
# %pip install -q pandas gensim pyLDAvis matplotlib


In [2]:
from pathlib import Path
from urllib.request import urlretrieve
import pandas as pd

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
DATA_FILE = DATA_DIR / "state-of-the-union.csv"
DATA_URL = "https://nyc3.digitaloceanspaces.com/ml-files-distro/v1/text-analysis/data/state-of-the-union.csv"

if not DATA_FILE.exists():
    urlretrieve(DATA_URL, DATA_FILE)

df = pd.read_csv(DATA_FILE)
assert "content" in df.columns, "Expected a 'content' column in the CSV."
df = df.dropna(subset=["content"]).reset_index(drop=True)
print(f"Documents: {len(df):,}")
print("Columns:", list(df.columns))
display(df.head(3))

Documents: 226
Columns: ['year', 'content']


,year,content
0,1790,"George Washington\nJanuary 8, 1790\n\nFellow-C..."
1,1790,\nState of the Union Address\nGeorge Washingto...
2,1791,\nState of the Union Address\nGeorge Washingto...


### Before modeling: what is in our corpus?

LDA discovers recurring word co-occurrence patterns; it does **not** know the subject matter in advance. Looking at the documents helps us judge whether the topics it finds make sense.

**Think about it:** Would you expect a presidential speech to cover only one issue? Which broad issues might recur across speeches?

In [3]:
df["word_count"] = df["content"].str.split().str.len()
display(df["word_count"].describe().round(0).to_frame("Words per document"))
print("Excerpt from first speech:\n")
print(df.loc[0, "content"][:900])

,Words per document
count,226.0
mean,7721.0
std,5495.0
min,990.0
25%,3936.0
50%,6082.0
75%,9607.0
max,33574.0


Excerpt from first speech:

George Washington
January 8, 1790

Fellow-Citizens of the Senate and House of Representatives:

I embrace with great satisfaction the opportunity which now presents itself
of congratulating you on the present favorable prospects of our public
affairs. The recent accession of the important state of North Carolina to
the Constitution of the United States (of which official information has
been received), the rising credit and respectability of our country, the
general and increasing good will toward the government of the Union, and
the concord, peace, and plenty with which we are blessed are circumstances
auspicious in an eminent degree to our national prosperity.

In resuming your consultations for the general good you can not but derive
encouragement from the reflection that the measures of the last session
have been as satisfactory to your constituents as the novelty and
difficulty of t


## 2. Preprocess text

We use Gensim's `simple_preprocess` to lowercase and tokenize text, discarding punctuation and short tokens. Then we remove common stopwords. This is a **choice**, not an absolute rule: sometimes words removed by a generic stopword list are important for a particular domain.

Unlike the original notebook, we explicitly remove stopwords before creating the dictionary so common function words are less likely to dominate the topics.

In [4]:
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS

raw_tokens = df["content"].apply(simple_preprocess)
texts = raw_tokens.apply(lambda tokens: [w for w in tokens if w not in STOPWORDS])

print("Before stopword removal:", raw_tokens.iloc[0][:35])
print("After stopword removal: ", texts.iloc[0][:35])

Before stopword removal: ['george', 'washington', 'january', 'fellow', 'citizens', 'of', 'the', 'senate', 'and', 'house', 'of', 'representatives', 'embrace', 'with', 'great', 'satisfaction', 'the', 'opportunity', 'which', 'now', 'presents', 'itself', 'of', 'congratulating', 'you', 'on', 'the', 'present', 'favorable', 'prospects', 'of', 'our', 'public', 'affairs', 'the']
After stopword removal:  ['george', 'washington', 'january', 'fellow', 'citizens', 'senate', 'house', 'representatives', 'embrace', 'great', 'satisfaction', 'opportunity', 'presents', 'congratulating', 'present', 'favorable', 'prospects', 'public', 'affairs', 'recent', 'accession', 'important', 'state', 'north', 'carolina', 'constitution', 'united', 'states', 'official', 'information', 'received', 'rising', 'credit', 'respectability', 'country']


**Think about it:** Which words disappeared? Could removing too many words make topics *less* meaningful? For a more controlled comparison later, you can rerun the following steps with `texts = raw_tokens` and see what changes.

## 3. Convert documents to bag-of-words

LDA uses word counts, not the original word order. Gensim's `Dictionary` assigns each retained word a numeric ID; `doc2bow` converts each document to `(word_id, count)` pairs.

The filtering parameters are worth understanding:
- `no_below=5`: remove words appearing in fewer than 5 documents.
- `no_above=0.5`: remove words appearing in more than 50% of documents.
- `keep_n=2000`: keep at most 2,000 vocabulary items (ranked by frequency after filtering).

These values are **demonstration settings**, not universal best practices. With a small corpus, `no_below=5` may remove useful rare terms.

In [5]:
from gensim import corpora

dictionary = corpora.Dictionary(texts)
dictionary.filter_extremes(no_below=5, no_above=0.5, keep_n=2000)
corpus = [dictionary.doc2bow(tokens) for tokens in texts]

print(f"Number of documents: {len(corpus)}")
print(f"Vocabulary size after filtering: {len(dictionary)}")
print("First 12 (word ID, count) pairs:", corpus[0][:12])
print("Decoded example:", [(dictionary[word_id], count) for word_id, count in corpus[0][:12]])

Number of documents: 226
Vocabulary size after filtering: 2000
First 12 (word ID, count) pairs: [(0, 1), (1, 1), (2, 1), (3, 1), (4, 1), (5, 1), (6, 1), (7, 1), (8, 1), (9, 1), (10, 2), (11, 1)]
Decoded example: [('according', 1), ('admitted', 1), ('advancement', 1), ('affording', 1), ('agree', 1), ('allowed', 1), ('arrangements', 1), ('ascertained', 1), ('attended', 1), ('blessed', 1), ('blessings', 2), ('communication', 1)]


**Connect to earlier lessons:** A BoW representation records **frequency**, not the position of each word. What information about the original speech has been lost?

## 4. Train an LDA model

**Latent Dirichlet Allocation (LDA)** represents each topic as a probability distribution over words. It also represents each document as a probability distribution over topics. We must specify the number of topics (`num_topics`) before training.

For fairer comparisons, we keep the training settings and random seed fixed when varying the topic count. `passes=10` means the algorithm makes multiple passes over the corpus; it is **not** the number of topics.

In [6]:
from gensim import models

RANDOM_STATE = 42

def train_lda(num_topics):
    return models.LdaModel(
        corpus=corpus,
        id2word=dictionary,
        num_topics=num_topics,
        passes=10,
        random_state=RANDOM_STATE
    )

def show_topics(model, topn=8):
    for topic_id in range(model.num_topics):
        terms = model.show_topic(topic_id, topn=topn)
        print(f"Topic {topic_id}: " + ", ".join(f"{word} ({weight:.3f})" for word, weight in terms))

lda_2 = train_lda(2)
show_topics(lda_2)

Topic 0: mexico (0.005), convention (0.003), spain (0.003), minister (0.003), tariff (0.003), court (0.003), gold (0.002), cent (0.002)
Topic 1: help (0.009), program (0.008), americans (0.008), budget (0.007), million (0.006), programs (0.005), today (0.005), billion (0.005)


### Interpreting the output

Each line shows the **most probable words within a topic** and their weights. A topic is **not** automatically given a human-readable name such as “Economy” or “Foreign Policy.” We infer labels by examining keywords and actual documents.

**Think about it:** Can you give each of the two topics a meaningful label? Are they perhaps too broad? A high-probability word is *characteristic of the topic*, but the displayed top words alone do not prove that the topic is coherent.

## 5. Compare different numbers of topics

We will compare **2, 5, and 15 topics**, as in the original demo. Before running, predict what might happen:

- **2 topics:** broad themes, potentially combining unrelated issues.
- **5 topics:** possibly easier to distinguish, but not guaranteed.
- **15 topics:** more specific themes, potentially repetitive or noisy.

There is no guarantee that more topics means a better model.

In [7]:
lda_5 = train_lda(5)
print("=== 5 topics ===")
show_topics(lda_5)

=== 5 topics ===
Topic 0: mexico (0.008), minister (0.004), convention (0.004), intercourse (0.004), spain (0.004), objects (0.003), france (0.003), bank (0.003)
Topic 1: program (0.016), million (0.009), billion (0.007), programs (0.007), problems (0.006), budget (0.006), farm (0.005), major (0.005)
Topic 2: americans (0.015), help (0.013), tonight (0.010), jobs (0.009), budget (0.009), today (0.007), programs (0.006), percent (0.006)
Topic 3: court (0.005), industrial (0.004), tariff (0.004), conference (0.004), corporations (0.004), islands (0.003), board (0.003), cent (0.003)
Topic 4: gold (0.006), silver (0.005), cent (0.004), cuba (0.003), bonds (0.003), spain (0.003), products (0.003), islands (0.003)


In [8]:
lda_15 = train_lda(15)
print("=== 15 topics ===")
show_topics(lda_15)

=== 15 topics ===
Topic 0: mexico (0.004), americans (0.002), minister (0.002), intercourse (0.002), tonight (0.002), ports (0.002), court (0.002), tariff (0.002)
Topic 1: program (0.022), million (0.014), billion (0.009), housing (0.008), farm (0.008), management (0.008), programs (0.008), prices (0.007)
Topic 2: fighting (0.011), soviet (0.009), enemy (0.007), today (0.007), allies (0.007), attack (0.006), task (0.006), air (0.006)
Topic 3: ships (0.008), mean (0.006), profitable (0.006), domain (0.006), speak (0.006), convention (0.005), friendship (0.005), constructive (0.005)
Topic 4: corporations (0.008), canal (0.006), panama (0.005), islands (0.005), industrial (0.005), courts (0.005), court (0.005), merely (0.004)
Topic 5: objects (0.005), intercourse (0.005), france (0.004), tribes (0.004), execution (0.004), indians (0.004), navigation (0.004), ports (0.003)
Topic 6: tariff (0.007), agricultural (0.006), cent (0.006), court (0.006), board (0.005), banks (0.005), conference (

**Think about it:** Which model seems easiest to interpret? Do some topics share many words? Would a topic label be defensible based on the actual documents, or only on a few keywords?

**Important:** Topic numbers are arbitrary identifiers. Topic 0 in the two-topic model is **not** necessarily the same theme as Topic 0 in the five-topic model.

## 6. Documents contain mixtures of topics

A common misconception is that LDA simply assigns one topic to each document. Instead, it estimates a **topic distribution** for each document. We will inspect the first speech using the five-topic model.

In [9]:
doc_id = 0
model = lda_5

topic_distribution = model.get_document_topics(corpus[doc_id], minimum_probability=0)
results = pd.DataFrame(topic_distribution, columns=["Topic", "Probability"])
display(results.sort_values("Probability", ascending=False).style.format({"Probability": "{:.3f}"}))
print("Sum of topic probabilities:", round(results["Probability"].sum(), 4))
print("Speech excerpt:\n", df.loc[doc_id, "content"][:600])

,Topic,Probability
0,0,0.994
2,2,0.001
4,4,0.001
3,3,0.001
1,1,0.001


Sum of topic probabilities: 1.0
Speech excerpt:
 George Washington
January 8, 1790

Fellow-Citizens of the Senate and House of Representatives:

I embrace with great satisfaction the opportunity which now presents itself
of congratulating you on the present favorable prospects of our public
affairs. The recent accession of the important state of North Carolina to
the Constitution of the United States (of which official information has
been received), the rising credit and respectability of our country, the
general and increasing good will toward the government of the Union, and
the concord, peace, and plenty with which we are blessed are cir


**Think about it:** Is one topic dominant? Are several topics represented? Do the prominent topic keywords seem consistent with the speech excerpt? The probabilities are **model estimates**, not ground-truth labels.

## 7. Visualize topics with pyLDAvis

The interactive display helps explore the topic model:
- Each circle represents a topic; **larger circles** generally represent more prevalent topics.
- The distances between circles approximate how different the topics are in a reduced two-dimensional space. **Overlap suggests similarity**, but the plot is only an approximation.
- Selecting a circle shows its prominent terms. The relevance slider changes how terms are ranked.

We visualize the five-topic model first. The display may take a moment to prepare.

In [10]:
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis

vis = gensimvis.prepare(lda_5, corpus, dictionary, sort_topics=False)
pyLDAvis.display(vis)

**Think about it:** Which topics look most distinct? Are some circles close together? Do their keyword lists explain why? Can a topic be visually distinct yet difficult to label meaningfully?

**Note:** The plot may reorder topics by prevalence unless `sort_topics=False` is used. We disable sorting here to make topic IDs easier to compare with printed output.

## 8. Wrap-up: how should we judge a topic model?

A useful topic model is not necessarily the one with the most topics. Interpretation should consider **topic coherence, overlap, stability, and whether the topics help answer the analytical question**. You may also examine representative documents, compare settings, or use coherence metrics in a more advanced exercise.

**Reflection (no submission):** If you applied LDA to 1,000 news articles, how would you decide whether the topics are meaningful? Why might the same article contain more than one topic?

### Key takeaways

1. LDA discovers patterns of word co-occurrence; it does not understand topics as a human would.
2. A topic is a distribution over words, and a document is a mixture of topics.
3. Preprocessing and the number of topics affect results.
4. Human interpretation is essential; topic models can produce overlapping or difficult-to-interpret themes.

**Optional exploration:** Change `doc_id`, try a different topic count in the visualization, or compare preprocessing with and without stopwords. These are optional ways to explore the demo, **not graded tasks**.